In [2]:
#导入库
import json
import numpy as np

#定义坐标类，封装各类计算及转化
class CoordinateSys:

    def __init__(self,axis_vectors):
        #转化为列向量
        self.basis_matrix=np.array(axis_vectors,dtype=np.float64).T
        #获取维度（方阵）
        self.dimension=self.basis_matrix.shape[0]


        #主动抛出异常
        #1.检查是否为方阵
        if self.basis_matrix.shape[0]!=self.basis_matrix.shape[1]:
            raise ValueError("Non square matrix!")

        #2.检查是否满秩，即是否能构成坐标系
        if np.linalg.matrix_rank(self.basis_matrix)!=self.dimension:
            raise ValueError("Unable to form a coordinate system!")


        #计算逆开销较大，此处预计算方便后续直接调用
        self.basis_inv=np.linalg.inv(self.basis_matrix)

        #初始化存放列表，先存放vectors，后更新为新坐标系下的新坐标
        self.vectors=[]

    #定义添加向量到self.vectors的函数
    def add_vector(self,vec):

        vector=np.array(vec,dtype=np.float64)


        #检查向量维度
        if len(vector)!=self.dimension:
            raise ValueError("Vector diemension mismatch!")

        self.vectors.append(vector)

    #计算绝对坐标
    def to_absolute(self,vec):

        return self.basis_matrix@vec

    #坐标系转移
    def change_axis(self,new_axis_vectors):

        #初始化坐标系
        new_coordinate=CoordinateSys(new_axis_vectors)

        #遍历所有向量，完成坐标转移
        for vec in self.vectors:

            abs_vec=self.to_absolute(vec)

            new_vec=new_coordinate.basis_inv@abs_vec

            new_coordinate.add_vector(new_vec)

        return new_coordinate


    #计算投影长度
    #直接取新坐标系下向量各分量的绝对值就是投影长度
    def axis_projection(self):  
        
        projection_results = []
        
        for vec in self.vectors:
            proj_lengths = np.abs(vec).tolist()
            projection_results.append(proj_lengths)
            
        return projection_results

    #计算夹角
    def axis_angle(self):
        angle_results = []
        for vec in self.vectors: 
            #计算向量模长
            vec_norm = np.linalg.norm(vec)
            #零向量特殊处理
            if vec_norm < 1e-12:
                angle_results.append([0.0 for _ in range(self.dimension)])
                continue
            #分量/模长求cos
            cos_thetas = vec / vec_norm  
            cos_thetas = np.clip(cos_thetas, -1.0, 1.0)  #防止浮点误差超范围
            angles = np.arccos(cos_thetas).tolist()      #arccos转弧度
            
            angle_results.append(angles)
            
        return angle_results


    #计算缩放，即计算行列式
    def cal_scale(self):

        det_value=np.linalg.det(self.basis_matrix)

        return float(np.abs(det_value))


def process_single_group(group_data):
    group_name = group_data["group_name"]
    ori_axis = group_data["ori_axis"]
    vectors = group_data["vectors"]
    tasks = group_data["tasks"]

    # 初始化初始坐标系
    try:
        cs = CoordinateSystem(ori_axis)
    except Exception as e:
        return {"group_name": group_name, "error": f"初始坐标系初始化失败：{str(e)}"}

   
    for vec in vectors:
        try:
            cs.add_vector(vec)
        except Exception as e:
            return {"group_name": group_name, "error": f"添加向量失败：{str(e)}"}

    # 执行任务（坐标变换+投影+夹角+面积）
    task_results = []
    for task in tasks:
        task_type = task["type"]
        task_result = {"task_type": task_type}
        try:
            if task_type == "change_axis":
                obj_axis = task["obj_axis"]
                cs = cs.change_axis(obj_axis)  # 执行坐标变换，得到新坐标系
                task_result["status"] = "success"
                task_result["new_axis"] = obj_axis
            elif task_type == "axis_projection":
                proj = cs.axis_projection()  # 调用你的投影计算逻辑
                task_result["status"] = "success"
                task_result["projection_lengths"] = proj
            elif task_type == "axis_angle":
                angle = cs.axis_angle()
                task_result["status"] = "success"
                task_result["angle_radians"] = angle
            elif task_type == "area":
                scale = cs.area_scale()
                task_result["status"] = "success"
                task_result["area_scale"] = scale
            else:
                task_result["status"] = "failed"
                task_result["error"] = f"未知任务类型：{task_type}"
        except Exception as e:
            task_result["status"] = "failed"
            task_result["error"] = str(e)
        task_results.append(task_result)

    return {
        "group_name": group_name,
        "dimension": cs.dimension,
        "final_vectors": [vec.tolist() for vec in cs.vectors],  # 最终的线性组合向量
        "task_results": task_results
    }

def main():
   
    input_file = "C:\\Users\\Sylvain\\Desktop\\QG\\data(1).json"
    output_file = "C:\\Users\\Sylvain\\Desktop\\QG\\result.json"

    try:
        with open(input_file, "r", encoding="utf-8") as f:
            all_groups = json.load(f)
    except FileNotFoundError:
        print(f"错误：未找到文件 {input_file}，请检查路径是否正确")
        return
    except json.JSONDecodeError:
        print(f"错误：{input_file} 不是合法的JSON文件")
        return

    all_results = []
    for group in all_groups:
        group_result = process_single_group(group)
        all_results.append(group_result)
        print(f"已完成：{group_result['group_name']}")

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)
    print(f"\n所有任务执行完成！结果已保存到：{output_file}")

if __name__ == "__main__":
    main()

已完成：2d_task_1
已完成：2d_task_2
已完成：2d_task_3
已完成：2d_task_4
已完成：2d_task_5
已完成：3d_task_1
已完成：3d_task_2
已完成：3d_task_3
已完成：3d_task_4
已完成：3d_task_5
已完成：8d_task_1
已完成：9d_task_1
已完成：10d_task_1
已完成：8d_task_2
已完成：10d_task_2

所有任务执行完成！结果已保存到：C:\Users\Sylvain\Desktop\QG\result.json
